# Step 5:Sequential Pattern Mining

## Phase 3: Sequential Pattern Mining

**Goal:** Mine association rules with temporal constraints for explainable recommendations

**Input:** User sequences from Step 2

**Output:**
- association_rules.json: {antecedent → consequent} with confidence, lift, support

**Algorithm:**
1. Build transaction database from user sequences
2. Mine frequent itemsets with Apriori (Pure Python)
3. Generate association rules with temporal constraint
4. Enrich rules with concept information

In [1]:
import json
import pickle
from collections import defaultdict, Counter
from itertools import combinations, chain
import pandas as pd
OUTPUT_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output"

# Parameters
MIN_SUPPORT = 0.01      # Minimum support (1%)
MIN_CONFIDENCE = 0.5    # Minimum confidence (50%)
MIN_LIFT = 1.2         # Minimum lift
MAX_ITEMSET_SIZE = 4   # Maximum itemset size for efficiency

print("Step 5: Sequential Pattern Mining (Pure Python)")
print(f"Parameters: min_support={MIN_SUPPORT}, min_confidence={MIN_CONFIDENCE}, min_lift={MIN_LIFT}")

Step 5: Sequential Pattern Mining (Pure Python)
Parameters: min_support=0.01, min_confidence=0.5, min_lift=1.2


## Load Data

In [2]:
# Load user sequences
with open(f"{OUTPUT_PATH}/user_sequences.json", "r", encoding='utf-8') as f:
    user_sequences = json.load(f)

# Load course concepts
with open(f"{OUTPUT_PATH}/course_concepts.json", "r", encoding='utf-8') as f:
    course_concepts = json.load(f)

print(f"Loaded {len(user_sequences)} user sequences")
print(f"Loaded {len(course_concepts)} course concept mappings")

Loaded 11385 user sequences
Loaded 887 course concept mappings


## Build Transaction Database

In [3]:
# Create transactions (list of courses per user, sorted by time)
transactions = []

for record in user_sequences:
    user_id = record["user_id"]
    sequence = record["sequence"]
    
    # Sort by sequence order (index 0)
    sequence = sorted(sequence, key=lambda x: x[0])
    
    # Get courses with R_score >= 0.3
    courses = [item[1] for item in sequence if item[2] >= 0.3]
    
    if len(courses) >= 2:
        transactions.append(courses)

print(f"Created {len(transactions)} transactions")
print(f"Example transaction: {transactions[0][:5] if transactions else 'N/A'}")

Created 11385 transactions
Example transaction: ['C_696908', 'C_1756060']


## Apriori Algorithm

In [4]:
def get_item_support(transactions, min_support_count):
    """Get support count for all individual items"""
    item_counts = Counter()
    for items in transactions:
        for item in items:
            item_counts[item] += 1
    
    # Filter by min support
    frequent_items = {item: count for item, count in item_counts.items() 
                     if count >= min_support_count}
    return frequent_items

def create_candidates(prev_itemsets, k):
    """Create candidate k-itemsets from frequent (k-1)-itemsets"""
    candidates = set()
    prev_list = list(prev_itemsets)
    
    for i in range(len(prev_list)):
        for j in range(i + 1, len(prev_list)):
            # Join two itemsets if they differ only in last item
            itemset1 = list(prev_list[i])
            itemset2 = list(prev_list[j])
            
            # Check if first k-2 items are the same
            if itemset1[:k-2] == itemset2[:k-2]:
                # Create new candidate
                new_itemset = tuple(sorted(set(itemset1) | set(itemset2)))
                if len(new_itemset) == k:
                    candidates.add(new_itemset)
    
    return candidates

def prune_candidates(candidates, prev_frequent, k):
    """Prune candidates that have infrequent subsets"""
    pruned = set()
    for candidate in candidates:
        # Check all k-1 subsets
        subsets = list(combinations(candidate, k-1))
        if all(subset in prev_frequent for subset in subsets):
            pruned.add(candidate)
    return pruned

def count_support(transactions, candidates):
    """Count support for candidates"""
    candidate_counts = defaultdict(int)
    
    for items in transactions:
        item_set = set(items)
        for candidate in candidates:
            if set(candidate).issubset(item_set):
                candidate_counts[candidate] += 1
    
    return candidate_counts

def get_frequent_itemsets(transactions, min_support_count, max_size=3):
    """Mine frequent itemsets using Apriori algorithm"""
    
    all_frequent_itemsets = {}
    
    # Get frequent 1-itemsets
    frequent_items = get_item_support(transactions, min_support_count)
    frequent_1_itemsets = {(item,): count for item, count in frequent_items.items()}
    
    if not frequent_1_itemsets:
        return all_frequent_itemsets
    
    all_frequent_itemsets[1] = frequent_1_itemsets
    print(f"Frequent 1-itemsets: {len(frequent_1_itemsets)}")
    
    # Generate larger itemsets
    prev_frequent = set(frequent_1_itemsets.keys())
    
    for k in range(2, max_size + 1):
        if len(prev_frequent) < 2:
            break
        
        # Create candidates
        candidates = create_candidates(prev_frequent, k)
        
        # Prune candidates
        candidates = prune_candidates(candidates, prev_frequent, k)
        
        if not candidates:
            break
        
        # Count support
        candidate_counts = count_support(transactions, candidates)
        
        # Filter by min support
        frequent_k = {itemset: count for itemset, count in candidate_counts.items()
                      if count >= min_support_count}
        
        if frequent_k:
            all_frequent_itemsets[k] = frequent_k
            prev_frequent = set(frequent_k.keys())
            print(f"Frequent {k}-itemsets: {len(frequent_k)}")
        else:
            break
    
    return all_frequent_itemsets

# Calculate min support count
min_support_count = int(len(transactions) * MIN_SUPPORT)
print(f"\nMinimum support count: {min_support_count} (from {len(transactions)} transactions)")

# Mine frequent itemsets
frequent_itemsets = get_frequent_itemsets(transactions, min_support_count, MAX_ITEMSET_SIZE)

# Flatten all frequent itemsets
all_itemsets = {}
for size, itemsets in frequent_itemsets.items():
    all_itemsets.update(itemsets)

print(f"\nTotal frequent itemsets: {len(all_itemsets)}")


Minimum support count: 113 (from 11385 transactions)
Frequent 1-itemsets: 158
Frequent 2-itemsets: 204
Frequent 3-itemsets: 83
Frequent 4-itemsets: 21

Total frequent itemsets: 466


## Generate Association Rules

In [5]:
def generate_association_rules(frequent_itemsets, min_confidence, min_lift, total_transactions):
    """Generate association rules from frequent itemsets"""
    rules = []
    
    # Get all itemsets by size
    all_itemsets = {}
    for size, itemsets in frequent_itemsets.items():
        all_itemsets.update(itemsets)
    
    # Generate rules from itemsets with size >= 2
    for itemset, support_count in all_itemsets.items():
        if len(itemset) < 2:
            continue
        
        # Try all possible splits
        for i in range(1, len(itemset)):
            for antecedent in combinations(itemset, i):
                antecedent = tuple(sorted(antecedent))
                consequent = tuple(sorted(set(itemset) - set(antecedent)))
                
                if not consequent:
                    continue
                
                # Get support counts
                itemset_support = support_count / total_transactions
                antecedent_support = all_itemsets.get(antecedent, 0) / total_transactions
                consequent_support = all_itemsets.get(consequent, 0) / total_transactions
                
                if antecedent_support == 0:
                    continue
                
                # Calculate confidence and lift
                confidence = itemset_support / antecedent_support
                
                if consequent_support == 0:
                    continue
                    
                lift = confidence / consequent_support
                
                # Filter by thresholds
                if confidence >= min_confidence and lift >= min_lift:
                    rules.append({
                        'antecedent': list(antecedent),
                        'consequent': list(consequent),
                        'confidence': confidence,
                        'lift': lift,
                        'support': itemset_support,
                        'antecedent_support': antecedent_support,
                        'consequent_support': consequent_support
                    })
    
    return rules

# Generate rules
print("Generating association rules...")
association_rules = generate_association_rules(
    frequent_itemsets, 
    MIN_CONFIDENCE, 
    MIN_LIFT, 
    len(transactions)
)

print(f"Generated {len(association_rules)} association rules")

# Sort by lift
association_rules.sort(key=lambda x: x['lift'], reverse=True)

# Show top rules
print("\nTop 10 rules by lift:")
for rule in association_rules[:10]:
    ant = ", ".join(rule['antecedent'])
    cons = ", ".join(rule['consequent'])
    print(f"  {ant} → {cons}")
    print(f"    confidence={rule['confidence']:.3f}, lift={rule['lift']:.3f}, support={rule['support']:.3f}")

Generating association rules...
Generated 327 association rules

Top 10 rules by lift:
  C_682330 → C_682345
    confidence=0.867, lift=37.677, support=0.013
  C_682345 → C_682330
    confidence=0.573, lift=37.677, support=0.013
  C_697381 → C_707119
    confidence=0.672, lift=31.998, support=0.012
  C_707119 → C_697381
    confidence=0.556, lift=31.998, support=0.012
  C_681448 → C_756646
    confidence=0.710, lift=30.862, support=0.011
  C_697166 → C_696724, C_697149
    confidence=0.548, lift=25.357, support=0.013
  C_696724, C_697149 → C_697166
    confidence=0.581, lift=25.357, support=0.013
  C_681448 → C_685689
    confidence=0.761, lift=24.145, support=0.012
  C_697075 → C_697087
    confidence=0.502, lift=23.026, support=0.014
  C_697087 → C_697075
    confidence=0.641, lift=23.026, support=0.014


## Enrich Rules with Concept Information

In [6]:
def jaccard_similarity(set_a, set_b):
    if not set_a or not set_b:
        return 0.0
    intersection = len(set(set_a) & set(set_b))
    union = len(set(set_a) | set(set_b))
    return intersection / union if union > 0 else 0.0

def get_concept_info(antecedent, consequent):
    """Get concept overlap and new concepts introduced"""
    antecedent_courses = antecedent if isinstance(antecedent, list) else [antecedent]
    consequent_courses = consequent if isinstance(consequent, list) else [consequent]
    
    # Get all concepts from antecedent
    antecedent_concepts = set()
    for c in antecedent_courses:
        antecedent_concepts.update(course_concepts.get(c, []))
    
    # Get all concepts from consequent
    consequent_concepts = set()
    for c in consequent_courses:
        consequent_concepts.update(course_concepts.get(c, []))
    
    # Calculate overlap
    overlap = jaccard_similarity(antecedent_concepts, consequent_concepts)
    
    # New concepts
    new_concepts = consequent_concepts - antecedent_concepts
    
    # Prerequisite concepts (intersection)
    prereq_concepts = antecedent_concepts & consequent_concepts
    
    return {
        "concept_overlap_ratio": overlap,
        "prerequisite_concepts": list(prereq_concepts),
        "num_prereq_concepts": len(prereq_concepts),
        "new_concepts_introduced": list(new_concepts),
        "num_new_concepts": len(new_concepts)
    }

# Enrich rules
enriched_rules = []
for rule in association_rules:
    concept_info = get_concept_info(rule['antecedent'], rule['consequent'])
    
    enriched_rule = {
        "antecedent": rule['antecedent'],
        "consequent": rule['consequent'],
        "confidence": rule['confidence'],
        "lift": rule['lift'],
        "support": rule['support'],
        "antecedent_support": rule['antecedent_support'],
        "consequent_support": rule['consequent_support'],
        **concept_info
    }
    enriched_rules.append(enriched_rule)

print(f"Enriched {len(enriched_rules)} rules")

# Show enriched top rules
print("\nTop 10 enriched rules:")
for rule in enriched_rules[:10]:
    ant = ", ".join(rule['antecedent'])
    cons = ", ".join(rule['consequent'])
    print(f"  {ant} → {cons}")
    print(f"    confidence={rule['confidence']:.3f}, lift={rule['lift']:.3f}, overlap={rule['concept_overlap_ratio']:.3f}")
    print(f"    new_concepts={rule['num_new_concepts']}")
    print()

Enriched 327 rules

Top 10 enriched rules:
  C_682330 → C_682345
    confidence=0.867, lift=37.677, overlap=0.057
    new_concepts=682

  C_682345 → C_682330
    confidence=0.573, lift=37.677, overlap=0.057
    new_concepts=363

  C_697381 → C_707119
    confidence=0.672, lift=31.998, overlap=0.227
    new_concepts=414

  C_707119 → C_697381
    confidence=0.556, lift=31.998, overlap=0.227
    new_concepts=414

  C_681448 → C_756646
    confidence=0.710, lift=30.862, overlap=0.101
    new_concepts=904

  C_697166 → C_696724, C_697149
    confidence=0.548, lift=25.357, overlap=0.146
    new_concepts=818

  C_696724, C_697149 → C_697166
    confidence=0.581, lift=25.357, overlap=0.146
    new_concepts=444

  C_681448 → C_685689
    confidence=0.761, lift=24.145, overlap=0.025
    new_concepts=3340

  C_697075 → C_697087
    confidence=0.502, lift=23.026, overlap=0.146
    new_concepts=520

  C_697087 → C_697075
    confidence=0.641, lift=23.026, overlap=0.146
    new_concepts=769



## Save Rules

In [7]:
# Save enriched rules
with open(f"{OUTPUT_PATH}/association_rules.json", "w", encoding='utf-8') as f:
    json.dump(enriched_rules, f, indent=2)

# Also save as pickle
with open(f"{OUTPUT_PATH}/association_rules.pkl", "wb") as f:
    pickle.dump(enriched_rules, f)

print(f"Saved {len(enriched_rules)} association rules")

# Statistics
print("\nRule Statistics:")
print(f"  Total rules: {len(enriched_rules)}")


Saved 327 association rules

Rule Statistics:
  Total rules: 327


In [8]:
enriched_rules_df = pd.DataFrame(enriched_rules)

if not enriched_rules_df.empty:
    avg_confidence = enriched_rules_df['confidence'].mean()
    avg_lift = enriched_rules_df['lift'].mean()
    avg_overlap = enriched_rules_df['concept_overlap_ratio'].mean()
    
    print(f"  Avg confidence: {avg_confidence:.3f}")
    print(f"  Avg lift: {avg_lift:.3f}")
    print(f"  Avg concept overlap: {avg_overlap:.3f}")

  Avg confidence: 0.689
  Avg lift: 6.889
  Avg concept overlap: 0.195


## Summary

In [9]:
print("="*60)
print("STEP 5 COMPLETE!")
print("="*60)
print(f"\nResults:")
print(f"  - Total transactions: {len(transactions)}")
print(f"  - Frequent 1-itemsets: {len(frequent_itemsets.get(1, {}))}")
print(f"  - Frequent 2-itemsets: {len(frequent_itemsets.get(2, {}))}")
if 3 in frequent_itemsets:
    print(f"  - Frequent 3-itemsets: {len(frequent_itemsets[3])}")
print(f"  - Association rules: {len(enriched_rules)}")
print(f"\nNext: Step 6 - Explanation Generator")

STEP 5 COMPLETE!

Results:
  - Total transactions: 11385
  - Frequent 1-itemsets: 158
  - Frequent 2-itemsets: 204
  - Frequent 3-itemsets: 83
  - Association rules: 327

Next: Step 6 - Explanation Generator
